# Kaggle Training — Light-CNN with LFCC Features
## ASVspoof 2019 Logical Access

**EDA-informed hyperparameters:**
- LFCC `[60, 251]`: 20 static + 20 Δ + 20 ΔΔ, n_fft=1024, hop=256, center=True
- Class imbalance 8.8:1 → WeightedRandomSampler + FocalLoss(α=0.75, γ=2.0)
- SpecAugment: T=30 frames (12%), F=10 bins (17%) — aggressive to improve generalization to unseen A07-A19 attacks
- Cosine Annealing Warm Restarts for curriculum learning across 30 epochs


In [ ]:
import os, glob

# Auto-detect dataset path (works for both common Kaggle dataset names)
POSSIBLE_ROOTS = [
    "/kaggle/input/asvpoof-2019-dataset",
    "/kaggle/input/asvpoof2019",
    "/kaggle/input/asvspoof-2019",
    "/kaggle/input/asvpoof-2019",
    "/kaggle/input/la-asvspoof2019",
]

DATA_ROOT = None
for p in POSSIBLE_ROOTS:
    if os.path.exists(p):
        DATA_ROOT = p
        break

if DATA_ROOT is None:
    # Try to find by searching
    matches = glob.glob("/kaggle/input/**/ASVspoof2019_LA_train", recursive=True)
    if matches:
        DATA_ROOT = matches[0].replace("/ASVspoof2019_LA_train", "")

if DATA_ROOT is None:
    raise RuntimeError(
        "Dataset not found. Please add the ASVspoof 2019 dataset to this notebook. "
        "Expected structure: /kaggle/input/<dataset-name>/ASVspoof2019_LA_train/flac/"
    )

print(f"Dataset root: {DATA_ROOT}")

# Verify expected directories
for split in ["ASVspoof2019_LA_train", "ASVspoof2019_LA_dev", "ASVspoof2019_LA_eval"]:
    path = os.path.join(DATA_ROOT, split, "flac")
    count = len(glob.glob(os.path.join(path, "*.flac"))) if os.path.exists(path) else 0
    print(f"  {split}/flac: {count} files")

PROTOCOL_DIR = None
for proto_name in ["ASVspoof2019_LA_cm_protocols", "protocols", "LA/ASVspoof2019_LA_cm_protocols"]:
    p = os.path.join(DATA_ROOT, proto_name)
    if os.path.exists(p):
        PROTOCOL_DIR = p
        break
if PROTOCOL_DIR:
    print(f"  Protocols: {PROTOCOL_DIR}")
else:
    print("  WARNING: Protocol directory not found, will search recursively")
    matches = glob.glob(os.path.join(DATA_ROOT, "**", "*.txt"), recursive=True)
    print(f"  Found .txt files: {matches[:5]}")


In [ ]:
import subprocess
subprocess.run(["pip", "install", "soundfile", "librosa", "-q"])


In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
import torch

CFG = {
    "sample_rate":       16000,
    "target_samples":    64000,      # 4.0s × 16kHz
    "pre_emphasis":      0.97,       # EDA: boost >3kHz vocoder artifact region
    "vad_top_db":        40,

    "n_fft":             1024,       # EDA: center=True → 251 frames from 64000 samples
    "hop_length":        256,
    "n_lfcc":            20,         # static → ×3 with deltas = 60 total
    "lfcc_frames":       251,        # confirmed by EDA notebook 07

    "batch_size":        128,        # GPU batch
    "epochs":            30,
    "lr":                1e-3,
    "lr_min":            1e-6,
    "weight_decay":      1e-4,
    "focal_alpha":       0.75,       # EDA: 8.8:1 imbalance, tune toward bonafide
    "focal_gamma":       2.0,
    "label_smoothing":   0.05,

    "spec_t_mask":       30,         # time mask width (frames)
    "spec_f_mask":       10,         # freq mask width (coefficients)
    "dropout":           0.3,

    "seed":              42,
    "device":            "cuda" if torch.cuda.is_available() else "cpu",
    "num_workers":       2,
    "output_dir":        "/kaggle/working",
    "model_name":        "light_cnn_lfcc",
}

torch.manual_seed(CFG["seed"])
import numpy as np; np.random.seed(CFG["seed"])

print(f"Device: {CFG['device']}")
if CFG["device"] == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# ── PROTOCOL PARSER ─────────────────────────────────────────────────────────
import pandas as pd, glob, os

def parse_protocols(data_root, protocol_dir=None):
    split_map = {
        "train": "ASVspoof2019_LA_train",
        "dev":   "ASVspoof2019_LA_dev",
        "eval":  "ASVspoof2019_LA_eval",
    }
    proto_patterns = {
        "train": ["*train*.txt", "*trn*.txt"],
        "dev":   ["*dev*.txt"],
        "eval":  ["*eval*.txt", "*evl*.txt"],
    }

    rows = []
    search_dirs = []
    if protocol_dir and os.path.exists(protocol_dir):
        search_dirs.append(protocol_dir)
    search_dirs += glob.glob(os.path.join(data_root, "**"), recursive=False)
    for root, dirs, files in os.walk(data_root):
        for f in files:
            if f.endswith(".txt") and "protocol" in f.lower():
                search_dirs.append(root)
                break

    for split, flac_subdir in split_map.items():
        flac_dir = os.path.join(data_root, flac_subdir, "flac")
        proto_file = None
        for sdir in set(search_dirs):
            for pat in proto_patterns[split]:
                matches = glob.glob(os.path.join(sdir, pat))
                if matches:
                    proto_file = matches[0]
                    break
            if proto_file:
                break

        if proto_file is None:
            print(f"WARNING: No protocol file found for {split}")
            continue

        with open(proto_file) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                speaker_id, audio_id, _, attack_id, key = parts[0], parts[1], parts[2], parts[3], parts[4]
                filepath = os.path.join(flac_dir, audio_id + ".flac")
                rows.append({
                    "speaker_id": speaker_id, "audio_id": audio_id,
                    "attack_id": attack_id, "key": key,
                    "is_spoof": 1 if key == "spoof" else 0,
                    "split": split, "file_path": filepath,
                    "file_exists": os.path.exists(filepath),
                })

    df = pd.DataFrame(rows)
    return df

manifest = parse_protocols(DATA_ROOT, PROTOCOL_DIR if 'PROTOCOL_DIR' in dir() else None)
print(f"Total rows: {len(manifest)}")
for split in ["train", "dev", "eval"]:
    sub = manifest[manifest.split == split]
    bon = len(sub[sub.key == "bonafide"])
    spf = len(sub[sub.key == "spoof"])
    missing = len(sub[~sub.file_exists])
    print(f"  {split}: {len(sub)} total | bon={bon} spoof={spf} ratio={spf/bon:.1f}:1 | missing={missing}")


In [ ]:
# ── AUDIO PREPROCESSING ─────────────────────────────────────────────────────
import numpy as np, soundfile as sf, librosa, scipy.fftpack as fft_

def load_audio(filepath, target_sr=16000):
    y, sr = sf.read(filepath)
    if y.ndim > 1:
        y = y.mean(axis=1)
    if sr != target_sr:
        y = librosa.resample(y, orig_sr=sr, target_sr=target_sr)
    return y.astype(np.float32), target_sr

def pre_emphasis(y, alpha=0.97):
    return np.concatenate([[y[0]], y[1:] - alpha * y[:-1]])

def vad_trim(y, top_db=40):
    intervals = librosa.effects.split(y=y, top_db=top_db)
    if len(intervals) == 0:
        return y
    trimmed = np.concatenate([y[s:e] for s, e in intervals])
    return trimmed if len(trimmed) > 1000 else y

def fix_length(y, target=64000, is_training=False):
    n = len(y)
    if n == target:
        return y
    if n > target:
        start = np.random.randint(0, n - target + 1) if is_training else (n - target) // 2
        return y[start:start + target]
    return np.pad(y, (0, target - n), mode="wrap")

def peak_normalize(y):
    return y / (np.max(np.abs(y)) + 1e-7)

def process_audio(filepath, is_training=False, cfg=CFG):
    y, sr = load_audio(filepath, cfg["sample_rate"])
    y = pre_emphasis(y, cfg["pre_emphasis"])
    y = vad_trim(y, cfg["vad_top_db"])
    y = fix_length(y, cfg["target_samples"], is_training)
    y = peak_normalize(y)
    return y, sr

def extract_lfcc(y, sr=16000, cfg=CFG):
    stft = librosa.stft(y, n_fft=cfg["n_fft"], hop_length=cfg["hop_length"], center=True)
    spectrum = np.abs(stft) ** 2
    n_bins = spectrum.shape[0]
    n_filters = cfg["n_lfcc"]
    fbank = np.zeros((n_filters, n_bins))
    pts = np.linspace(0, n_bins - 1, n_filters + 2, dtype=int)
    for i in range(n_filters):
        fbank[i, pts[i]:pts[i+1]] = np.linspace(0, 1, pts[i+1] - pts[i])
        fbank[i, pts[i+1]:pts[i+2]] = np.linspace(1, 0, pts[i+2] - pts[i+1])
    energy = np.dot(fbank, spectrum)
    log_energy = np.log(np.maximum(energy, 1e-8))
    static = fft_.dct(log_energy, type=2, axis=0, norm="ortho")[:n_filters]
    delta = librosa.feature.delta(static, order=1)
    delta2 = librosa.feature.delta(static, order=2)
    lfcc = np.vstack([static, delta, delta2])
    # Ensure exact [60, 251] shape
    T = cfg["lfcc_frames"]
    if lfcc.shape[1] < T:
        lfcc = np.pad(lfcc, ((0,0),(0, T - lfcc.shape[1])), mode="edge")
    else:
        lfcc = lfcc[:, :T]
    return lfcc.astype(np.float32)

# Verify on one sample
test_row = manifest[manifest.split == "train"].iloc[0]
y_test, sr_test = process_audio(test_row.file_path, is_training=True)
lfcc_test = extract_lfcc(y_test, sr_test)
print(f"Audio shape: {y_test.shape}, LFCC shape: {lfcc_test.shape}")
assert lfcc_test.shape == (60, 251), f"Expected (60, 251), got {lfcc_test.shape}"
print("Preprocessing verified.")


In [ ]:
# ── DATASET ─────────────────────────────────────────────────────────────────
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

class LFCCDataset(Dataset):
    def __init__(self, df, is_training=False, cfg=CFG):
        self.df = df.reset_index(drop=True)
        self.is_training = is_training
        self.cfg = cfg

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            y, sr = process_audio(row.file_path, self.is_training, self.cfg)
            x = extract_lfcc(y, sr, self.cfg)
        except Exception:
            x = np.zeros((60, self.cfg["lfcc_frames"]), dtype=np.float32)

        if self.is_training:
            # SpecAugment: time masking
            if np.random.rand() < 0.5:
                t = np.random.randint(0, self.cfg["spec_t_mask"])
                t0 = np.random.randint(0, max(1, self.cfg["lfcc_frames"] - t))
                x[:, t0:t0+t] = x.mean()
            # SpecAugment: frequency masking
            if np.random.rand() < 0.5:
                f = np.random.randint(0, self.cfg["spec_f_mask"])
                f0 = np.random.randint(0, max(1, 60 - f))
                x[f0:f0+f, :] = x.mean()

        return torch.from_numpy(x).unsqueeze(0), torch.tensor(int(row.is_spoof), dtype=torch.long)

train_df = manifest[manifest.split == "train"].reset_index(drop=True)
dev_df   = manifest[manifest.split == "dev"].reset_index(drop=True)

# Weighted sampler to handle 8.8:1 imbalance
labels = train_df.is_spoof.values
class_counts = np.bincount(labels)
class_weights = 1.0 / class_counts
sample_weights = torch.FloatTensor(class_weights[labels])
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_ds = LFCCDataset(train_df, is_training=True)
dev_ds   = LFCCDataset(dev_df, is_training=False)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], sampler=sampler,
                          num_workers=CFG["num_workers"], pin_memory=True)
dev_loader   = DataLoader(dev_ds, batch_size=CFG["batch_size"]*2, shuffle=False,
                          num_workers=CFG["num_workers"], pin_memory=True)

print(f"Train: {len(train_ds)} | Dev: {len(dev_ds)}")
print(f"Batches/epoch: {len(train_loader)} | Batch size: {CFG['batch_size']}")
print(f"Bonafide weight: {class_weights[0]:.4f} | Spoof weight: {class_weights[1]:.4f}")


In [ ]:
# ── MODEL: Light-CNN with Max-Feature-Map (MFM) ──────────────────────────────
# Architecture proven in: Wu et al. "Light CNN for Deep Face Representation"
# Adapted for ASVspoof by: LFCC-LCNN system (LFCC [60,251] → MFM conv → EER ~5%)

import torch.nn as nn, torch.nn.functional as F

class MaxFeatureMap2D(nn.Module):
    def forward(self, x):
        c = x.size(1)
        a, b = torch.split(x, c // 2, dim=1)
        return torch.max(a, b)

class LightCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 64, 5, 1, 2), MaxFeatureMap2D(),
            nn.MaxPool2d(2, 2)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, 1, 1, 0), MaxFeatureMap2D(),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 96, 3, 1, 1), MaxFeatureMap2D(),
            nn.MaxPool2d(2, 2), nn.BatchNorm2d(48)
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(48, 96, 1, 1, 0), MaxFeatureMap2D(),
            nn.BatchNorm2d(48),
            nn.Conv2d(48, 128, 3, 1, 1), MaxFeatureMap2D(),
            nn.MaxPool2d(2, 2)
        )
        self.block4 = nn.Sequential(
            nn.Conv2d(64, 128, 1, 1, 0), MaxFeatureMap2D(),
            nn.BatchNorm2d(64),
            nn.Conv2d(64, 64, 3, 1, 1), MaxFeatureMap2D(),
            nn.BatchNorm2d(32)
        )
        self.block5 = nn.Sequential(
            nn.Conv2d(32, 64, 1, 1, 0), MaxFeatureMap2D(),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 64, 3, 1, 1), MaxFeatureMap2D(),
            nn.MaxPool2d(2, 2)
        )
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, 64), nn.ReLU(),
            nn.Dropout(CFG["dropout"]),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)
        x = self.global_pool(x)
        return self.classifier(x)

device = torch.device(CFG["device"])
model = LightCNN().to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"LightCNN parameters: {n_params:,}")

# Verify forward pass
with torch.no_grad():
    dummy = torch.zeros(2, 1, 60, 251).to(device)
    out = model(dummy)
    print(f"Output shape: {out.shape}  (expected [2, 2])")


In [ ]:
# ── LOSS, OPTIMIZER, SCHEDULER ───────────────────────────────────────────────

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, reduction="none", label_smoothing=self.label_smoothing)
        pt = torch.exp(-ce)
        alpha_t = torch.where(targets == 1, self.alpha, 1.0 - self.alpha)
        return (alpha_t * (1 - pt) ** self.gamma * ce).mean()

def compute_eer(y_true, y_score):
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y_true, y_score, pos_label=1)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    return float((fpr[idx] + fnr[idx]) / 2)

criterion = FocalLoss(CFG["focal_alpha"], CFG["focal_gamma"], CFG["label_smoothing"])
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=CFG["lr_min"]
)
scaler = torch.cuda.amp.GradScaler(enabled=(CFG["device"] == "cuda"))
print("Optimizer, scheduler, scaler initialized.")


In [ ]:
# ── TRAINING LOOP ────────────────────────────────────────────────────────────
import time, json
from sklearn.metrics import roc_auc_score

best_eer = float("inf")
best_auc = 0.0
history = []
save_path = os.path.join(CFG["output_dir"], f"{CFG['model_name']}_best.pth")

for epoch in range(1, CFG["epochs"] + 1):
    model.train()
    total_loss, t0 = 0.0, time.time()

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(CFG["device"] == "cuda")):
            logits = model(xb)
            loss = criterion(logits, yb)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * xb.size(0)

    scheduler.step(epoch)
    train_loss = total_loss / len(train_ds)

    model.eval()
    all_probs, all_targets = [], []
    with torch.no_grad():
        for xb, yb in dev_loader:
            xb = xb.to(device)
            with torch.cuda.amp.autocast(enabled=(CFG["device"] == "cuda")):
                logits = model(xb)
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            all_probs.append(probs)
            all_targets.append(yb.numpy())

    y_prob = np.concatenate(all_probs)
    y_true = np.concatenate(all_targets)
    eer = compute_eer(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    acc = ((y_prob >= 0.5) == y_true).mean()
    elapsed = time.time() - t0

    print(f"Ep {epoch:02d}/{CFG['epochs']} | loss={train_loss:.4f} | EER={eer*100:.2f}% | AUC={auc:.4f} | acc={acc*100:.1f}% | {elapsed:.0f}s")

    history.append({"epoch": epoch, "train_loss": round(train_loss, 6),
                    "val_eer": round(eer, 6), "val_auc": round(auc, 6)})

    if eer < best_eer:
        best_eer = eer
        best_auc = auc
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "eer": best_eer,
            "auc": best_auc,
            "cfg": CFG,
        }, save_path)
        print(f"  >>> Best model saved: EER={best_eer*100:.2f}% AUC={best_auc:.4f}")

json.dump(history, open(os.path.join(CFG["output_dir"], f"{CFG['model_name']}_history.json"), "w"), indent=2)
print(f"\nTraining complete. Best EER: {best_eer*100:.2f}% | Best AUC: {best_auc:.4f}")


In [ ]:
# ── FINAL RESULTS & TRAINING CURVE ──────────────────────────────────────────
import matplotlib.pyplot as plt

epochs_list = [h["epoch"] for h in history]
eers  = [h["val_eer"]*100 for h in history]
aucs  = [h["val_auc"] for h in history]
losses = [h["train_loss"] for h in history]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(epochs_list, losses, color="#3498db", lw=2)
axes[0].set_title("Training Loss (Focal)")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")

axes[1].plot(epochs_list, eers, color="#e74c3c", lw=2, marker="o", markersize=4)
axes[1].axhline(min(eers), color="gray", linestyle="--", label=f"Best EER: {min(eers):.2f}%")
axes[1].set_title("Dev EER (%)")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("EER %"); axes[1].legend()

axes[2].plot(epochs_list, aucs, color="#2ecc71", lw=2, marker="o", markersize=4)
axes[2].axhline(max(aucs), color="gray", linestyle="--", label=f"Best AUC: {max(aucs):.4f}")
axes[2].set_title("Dev AUC")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("AUC"); axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(CFG["output_dir"], f"{CFG['model_name']}_training_curve.png"), dpi=150)
plt.show()

print(f"Model saved: {save_path}")
print(f"Best Dev EER: {best_eer*100:.2f}%")
print(f"Best Dev AUC: {best_auc:.4f}")
